# Notebook 10 — Chronological Validation Design (Version 2)

Define the purged expanding-window validation design for the locked Version 2
panel, whose reference quarter is $t$ and whose predictors are measured at
$t-1$. The target remains failure during $(t,t+365]$.

This notebook does **not** select predictors, fit preprocessing, tune a model,
choose a threshold, or compare final-test performance. It reads the approved
calendar boundaries from the shared Version 2 configuration, calculates the
realised sample after Notebook 09's exact-lag exclusions, proves that no
training outcome window overlaps its evaluation period, and writes row-level
membership keys for downstream modelling notebooks.

The 2020Q1–2025Q2 calendar was also examined in Version 1. That historical fact
is disclosed here, but Version 1 final-test performance is not an input to any
Version 2 decision.


In [ ]:
# Load the shared Version 2 contract and lock all paths

from pathlib import Path
import hashlib
import importlib.util
import json
import os

import numpy as np
import pandas as pd
from IPython.display import display


COLAB_MYDRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None

if COLAB_MYDRIVE.is_dir():
    print("Google Drive is already mounted at /content/drive.")
elif drive is not None:
    try:
        drive.mount("/content/drive", force_remount=False)
    except (ValueError, OSError) as exc:
        if COLAB_MYDRIVE.is_dir():
            print("Google Drive became available after the mount warning; continuing.")
        else:
            raise RuntimeError(
                "Google Drive could not be mounted. In Colab, open the Files pane, "
                "click Mount Drive, complete authorisation, and then rerun this cell."
            ) from exc


DEFAULT_PROJECT_ROOT = Path(
    os.environ.get("DISSERTATION_PROJECT_ROOT", "/content/drive/MyDrive/Data Final")
).expanduser()
explicit_config = os.environ.get("V2_CONFIG_PATH")
CONFIG_PATH = (
    Path(explicit_config).expanduser()
    if explicit_config
    else DEFAULT_PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
)
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Shared Version 2 config not found: {CONFIG_PATH}")

config_spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
config = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config)

REQUIRED_CONFIG_FIELDS = {
    "ANALYSIS_VERSION", "VERSION_2_MAIN_TIMING_POLICY", "V2_OUTPUT_ROOT",
    "VALIDATION_FOLDS", "FINAL_TEST_DESIGN", "TARGET_HORIZON_DAYS",
}
missing_config_fields = sorted(REQUIRED_CONFIG_FIELDS.difference(dir(config)))
if missing_config_fields:
    raise AttributeError(
        "Shared Version 2 config is missing required fields: "
        + ", ".join(missing_config_fields)
    )

ANALYSIS_VERSION = config.ANALYSIS_VERSION
TIMING_POLICY = dict(config.VERSION_2_MAIN_TIMING_POLICY)
VALIDATION_FOLDS = tuple(dict(item) for item in config.VALIDATION_FOLDS)
FINAL_TEST_DESIGN = dict(config.FINAL_TEST_DESIGN)
TARGET_HORIZON_DAYS = int(config.TARGET_HORIZON_DAYS)
V2_OUTPUT_ROOT = Path(config.V2_OUTPUT_ROOT)

PROJECT_ROOT = Path(
    getattr(config, "PROJECT_ROOT", CONFIG_PATH.parent.parent)
).expanduser()
V2_ROOT = Path(getattr(config, "V2_ROOT", CONFIG_PATH.parent)).expanduser()


def deduplicate_paths(paths):
    unique = []
    for path in paths:
        path = Path(path).expanduser()
        if path not in unique:
            unique.append(path)
    return unique


# Notebook 08 and 09 were executed during a path-convention transition. Resolve
# one complete V2 bundle without ever searching or reading the archived V1 tree.
candidate_version_roots = deduplicate_paths([
    V2_OUTPUT_ROOT / ANALYSIS_VERSION,
    V2_OUTPUT_ROOT,
    V2_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    V2_ROOT / "Notebook Outputs",
    PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    PROJECT_ROOT / "Notebook Outputs",
    DEFAULT_PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
    DEFAULT_PROJECT_ROOT / "Notebook Outputs",
])

bundle_relative_paths = {
    "final_panel": Path("09_Final_Modelling_Panel") / "v2_final_modelling_panel_2001Q2_2025Q2.parquet",
    "lag_match_audit": Path("09_Final_Modelling_Panel") / "v2_exact_bank_lag_match_audit.parquet",
    "sample_summary": Path("09_Final_Modelling_Panel") / "v2_sample_summary.csv",
    "feature_manifest": Path("09_Final_Modelling_Panel") / "v2_notebook09_feature_manifest.csv",
    "macro_lag_audit": Path("08_Macroeconomic_Data_Preparation") / "macro_predictors_lag_mapping_audit.csv",
}


def bundle_is_complete(root):
    return all((root / relative).is_file() for relative in bundle_relative_paths.values())


complete_bundle_roots = [root for root in candidate_version_roots if bundle_is_complete(root)]
if not complete_bundle_roots:
    # Narrow fallback: search only approved V2/current output roots for the
    # V2-specific panel filename, then require the entire bundle beside it.
    search_roots = deduplicate_paths([
        V2_OUTPUT_ROOT,
        V2_ROOT / "Notebook Outputs",
        PROJECT_ROOT / "Notebook Outputs",
        DEFAULT_PROJECT_ROOT / "Notebook Outputs",
    ])
    for search_root in search_roots:
        if not search_root.is_dir():
            continue
        for panel_candidate in search_root.rglob(
            "v2_final_modelling_panel_2001Q2_2025Q2.parquet"
        ):
            inferred_root = panel_candidate.parent.parent
            if inferred_root not in candidate_version_roots:
                candidate_version_roots.append(inferred_root)
            if bundle_is_complete(inferred_root) and inferred_root not in complete_bundle_roots:
                complete_bundle_roots.append(inferred_root)

if not complete_bundle_roots:
    checked = "\n".join(str(root) for root in candidate_version_roots)
    raise FileNotFoundError(
        "No complete Version 2 Notebook 08/09 handoff bundle was found. "
        "Checked these output roots:\n" + checked
    )

VERSION_OUTPUT_ROOT = complete_bundle_roots[0]
NB08_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "08_Macroeconomic_Data_Preparation"
NB09_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "09_Final_Modelling_Panel"
NB10_OUTPUT_DIR = VERSION_OUTPUT_ROOT / "10_Chronological_Validation_Design"
NB10_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PANEL_PATH = VERSION_OUTPUT_ROOT / bundle_relative_paths["final_panel"]
LAG_MATCH_AUDIT_PATH = VERSION_OUTPUT_ROOT / bundle_relative_paths["lag_match_audit"]
NB09_SAMPLE_SUMMARY_PATH = VERSION_OUTPUT_ROOT / bundle_relative_paths["sample_summary"]
NB09_FEATURE_MANIFEST_PATH = VERSION_OUTPUT_ROOT / bundle_relative_paths["feature_manifest"]
MACRO_LAG_AUDIT_PATH = VERSION_OUTPUT_ROOT / bundle_relative_paths["macro_lag_audit"]

required_inputs = [
    CONFIG_PATH, FINAL_PANEL_PATH, LAG_MATCH_AUDIT_PATH,
    NB09_SAMPLE_SUMMARY_PATH, NB09_FEATURE_MANIFEST_PATH, MACRO_LAG_AUDIT_PATH,
]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


input_panel_sha256_before = sha256_file(FINAL_PANEL_PATH)

cell_01_checks = {
    "The analysis is the Version 2 lagged-bank main design":
        ANALYSIS_VERSION == "v2_bank_lag1_main",
    "The bank predictor source is the exact preceding quarter":
        TIMING_POLICY.get("bank_predictor_source") == "exact_t_minus_1",
    "The macro predictor source is t-1":
        TIMING_POLICY.get("macro_predictor_source") == "t_minus_1",
    "The target remains anchored at t over 1-365 days":
        TIMING_POLICY.get("target_window") == "(t, t_plus_365_days]"
        and TARGET_HORIZON_DAYS == 365,
    "All three validation folds are present in shared config":
        len(VALIDATION_FOLDS) == 3,
    "The final-test design is present in shared config":
        set(FINAL_TEST_DESIGN) == {
            "train_start", "train_end", "purge_start", "purge_end",
            "test_start", "test_end",
        },
    "Version 1 remains archived and read only":
        TIMING_POLICY.get("archived_v1_read_only") is True,
    "Adoption of lagging is unconditional on performance":
        TIMING_POLICY.get("main_analysis_choice_is_unconditional_on_performance") is True,
    "One complete Version 2 Notebook 08/09 bundle was resolved":
        bundle_is_complete(VERSION_OUTPUT_ROOT),
    "All selected Version 2 handoffs exist": all(path.is_file() for path in required_inputs),
}
notebook_10_cell_01_gate = pd.DataFrame({
    "check": list(cell_01_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_01_checks.values()],
    "passed": list(cell_01_checks.values()),
})
notebook_10_cell_01_gate.to_csv(NB10_OUTPUT_DIR / "cell_01_setup_gate.csv", index=False)

print("Notebook 10 Cell 1 validation gate:")
display(notebook_10_cell_01_gate)
failed = [name for name, passed in cell_01_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 10 Cell 1 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 1 completed successfully.")
print("Resolved Version 2 handoff root:", VERSION_OUTPUT_ROOT)
print("Input panel:", FINAL_PANEL_PATH)
print("Output directory:", NB10_OUTPUT_DIR)


In [ ]:
# Load the locked Notebook 09 panel without changing any value

if not notebook_10_cell_01_gate["passed"].all():
    raise AssertionError("Cell 2 blocked because Cell 1 did not pass.")

panel = pd.read_parquet(FINAL_PANEL_PATH)
panel["report_date"] = pd.to_datetime(panel["report_date"], errors="raise").dt.normalize()
panel["failed_1yr"] = pd.to_numeric(panel["failed_1yr"], errors="raise").astype("int8")
panel = panel.sort_values(["report_date", "cert", "idrssd"], kind="mergesort").reset_index(drop=True)

feature_manifest = pd.read_csv(NB09_FEATURE_MANIFEST_PATH)
if "v2_notebook09_feature" not in feature_manifest.columns:
    raise KeyError("Notebook 09 feature manifest lacks v2_notebook09_feature.")
MODEL_FEATURES = feature_manifest["v2_notebook09_feature"].astype(str).tolist()
EXPECTED_PANEL_COLUMNS = ["report_date", "cert", "idrssd"] + MODEL_FEATURES + ["failed_1yr"]

lag_match_audit = pd.read_parquet(LAG_MATCH_AUDIT_PATH)
for column in ["report_date", "source_report_date", "expected_bank_source_date"]:
    lag_match_audit[column] = pd.to_datetime(lag_match_audit[column], errors="raise").dt.normalize()

nb09_sample_summary = pd.read_csv(
    NB09_SAMPLE_SUMMARY_PATH, parse_dates=["start", "end"]
)
macro_lag_audit = pd.read_csv(
    MACRO_LAG_AUDIT_PATH, parse_dates=["report_date", "source_quarter"]
)
macro_lag_audit["report_date"] = macro_lag_audit.report_date.dt.normalize()
macro_lag_audit["source_quarter"] = macro_lag_audit.source_quarter.dt.normalize()

sensitive_fields = {
    "failure_date", "fail_date", "days_to_failure", "target_reason",
    "target_window_end", "positive_horizon_days", "source_report_date",
    "expected_bank_source_date", "cert_source", "cert_reference",
}

cell_02_checks = {
    "Panel columns exactly match the locked Notebook 09 handoff":
        list(panel.columns) == EXPECTED_PANEL_COLUMNS,
    "Panel IDRSSD-quarter keys are unique":
        not panel.duplicated(["idrssd", "report_date"]).any(),
    "Panel CERT-quarter keys are unique":
        not panel.duplicated(["cert", "report_date"]).any(),
    "Every reference date is a quarter-end": panel.report_date.dt.is_quarter_end.all(),
    "The target is binary": set(panel.failed_1yr.unique()).issubset({0, 1}),
    "The feature manifest excludes identifiers, dates and the target":
        set(MODEL_FEATURES).isdisjoint({"report_date", "cert", "idrssd", "failed_1yr"}),
    "Audit-only failure and source fields are absent from the primary panel":
        sensitive_fields.isdisjoint(panel.columns),
    "Notebook 09 exact-lag audit has unique IDRSSD-quarter keys":
        not lag_match_audit.duplicated(["idrssd", "report_date"]).any(),
    "Notebook 08 macro-lag audit has one row per reference quarter":
        not macro_lag_audit.duplicated(["report_date"]).any(),
    "Every macro source is exactly one quarter before its reference date":
        macro_lag_audit.source_quarter.eq(
            macro_lag_audit.report_date - pd.offsets.QuarterEnd()
        ).all()
        and macro_lag_audit.quarter_index_gap.eq(1).all()
        and macro_lag_audit.macro_lag_applications.eq(1).all(),
    "Reading the panel did not change its file bytes":
        sha256_file(FINAL_PANEL_PATH) == input_panel_sha256_before,
}
notebook_10_cell_02_gate = pd.DataFrame({
    "check": list(cell_02_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_02_checks.values()],
    "passed": list(cell_02_checks.values()),
})
notebook_10_cell_02_gate.to_csv(NB10_OUTPUT_DIR / "cell_02_panel_gate.csv", index=False)

print("Notebook 10 Cell 2 validation gate:")
display(notebook_10_cell_02_gate)
print("\nLocked panel shape:", panel.shape)
print("Reference coverage:", panel.report_date.min().date(), "to", panel.report_date.max().date())
failed = [name for name, passed in cell_02_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 10 Cell 2 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 2 completed successfully.")


In [ ]:
# Render the shared config as the approved chronological schedule

if not notebook_10_cell_02_gate["passed"].all():
    raise AssertionError("Cell 3 blocked because Cell 2 did not pass.")


def quarter_end(label):
    return pd.Period(str(label), freq="Q").end_time.normalize()


def quarter_count(start, end):
    return int(pd.period_range(start=start, end=end, freq="Q").size)


EXPECTED_VALIDATION_FOLDS = (
    {
        "fold": 1, "train_start": "2001Q2", "train_end": "2008Q4",
        "purge_start": "2009Q1", "purge_end": "2009Q4",
        "validation_start": "2010Q1", "validation_end": "2012Q4",
    },
    {
        "fold": 2, "train_start": "2001Q2", "train_end": "2011Q4",
        "purge_start": "2012Q1", "purge_end": "2012Q4",
        "validation_start": "2013Q1", "validation_end": "2015Q4",
    },
    {
        "fold": 3, "train_start": "2001Q2", "train_end": "2014Q4",
        "purge_start": "2015Q1", "purge_end": "2015Q4",
        "validation_start": "2016Q1", "validation_end": "2018Q4",
    },
)
EXPECTED_FINAL_TEST_DESIGN = {
    "train_start": "2001Q2", "train_end": "2018Q4",
    "purge_start": "2019Q1", "purge_end": "2019Q4",
    "test_start": "2020Q1", "test_end": "2025Q2",
}

fold_definition_records = []
schedule_records = []
for item in VALIDATION_FOLDS:
    fold = int(item["fold"])
    stage = f"fold{fold}"
    train_start, train_end = quarter_end(item["train_start"]), quarter_end(item["train_end"])
    purge_start, purge_end = quarter_end(item["purge_start"]), quarter_end(item["purge_end"])
    eval_start, eval_end = quarter_end(item["validation_start"]), quarter_end(item["validation_end"])
    fold_definition_records.append({
        "stage": stage,
        "training_period": f'{item["train_start"]}–{item["train_end"]}',
        "purge_period": f'{item["purge_start"]}–{item["purge_end"]}',
        "evaluation_period": f'{item["validation_start"]}–{item["validation_end"]}',
        "evaluation_role": "validation",
    })
    schedule_records.extend([
        {"period": f"{stage}_train", "stage": stage, "role": "training", "start": train_start, "end": train_end},
        {"period": f"{stage}_purge", "stage": stage, "role": "purge", "start": purge_start, "end": purge_end},
        {"period": f"{stage}_validation", "stage": stage, "role": "validation", "start": eval_start, "end": eval_end},
    ])

final_train_start = quarter_end(FINAL_TEST_DESIGN["train_start"])
final_train_end = quarter_end(FINAL_TEST_DESIGN["train_end"])
final_purge_start = quarter_end(FINAL_TEST_DESIGN["purge_start"])
final_purge_end = quarter_end(FINAL_TEST_DESIGN["purge_end"])
final_test_start = quarter_end(FINAL_TEST_DESIGN["test_start"])
final_test_end = quarter_end(FINAL_TEST_DESIGN["test_end"])

fold_definition_records.append({
    "stage": "final",
    "training_period": f'{FINAL_TEST_DESIGN["train_start"]}–{FINAL_TEST_DESIGN["train_end"]}',
    "purge_period": f'{FINAL_TEST_DESIGN["purge_start"]}–{FINAL_TEST_DESIGN["purge_end"]}',
    "evaluation_period": f'{FINAL_TEST_DESIGN["test_start"]}–{FINAL_TEST_DESIGN["test_end"]}',
    "evaluation_role": "test",
})
schedule_records.extend([
    {"period": "final_train", "stage": "final", "role": "training", "start": final_train_start, "end": final_train_end},
    {"period": "final_purge", "stage": "final", "role": "purge", "start": final_purge_start, "end": final_purge_end},
    {"period": "final_test", "stage": "final", "role": "test", "start": final_test_start, "end": final_test_end},
])

fold_definitions = pd.DataFrame(fold_definition_records)
schedule = pd.DataFrame(schedule_records)
schedule["quarter_count"] = [quarter_count(row.start, row.end) for row in schedule.itertuples()]

schedule_by_stage = {
    stage: group.set_index("role") for stage, group in schedule.groupby("stage", sort=False)
}
train_ends = [quarter_end(item["train_end"]) for item in VALIDATION_FOLDS]

cell_03_checks = {
    "Shared config matches the three approved validation folds":
        VALIDATION_FOLDS == EXPECTED_VALIDATION_FOLDS,
    "Shared config matches the approved final out-of-time design":
        FINAL_TEST_DESIGN == EXPECTED_FINAL_TEST_DESIGN,
    "All chronological boundaries are reference quarter-ends":
        schedule.start.dt.is_quarter_end.all() and schedule.end.dt.is_quarter_end.all(),
    "Every purge contains exactly four reference quarters":
        schedule.loc[schedule.role.eq("purge"), "quarter_count"].eq(4).all(),
    "Every validation window contains exactly twelve reference quarters":
        schedule.loc[schedule.role.eq("validation"), "quarter_count"].eq(12).all(),
    "The final test contains the fixed twenty-two reference quarters":
        schedule.loc[schedule.period.eq("final_test"), "quarter_count"].eq(22).all(),
    "Training windows expand strictly across validation folds":
        all(later > earlier for earlier, later in zip(train_ends, train_ends[1:])),
    "Every stage is ordered training, purge, then evaluation": all(
        group.loc["training", "end"] < group.loc["purge", "start"]
        and group.loc["purge", "end"] < group.loc[
            "test" if stage == "final" else "validation", "start"
        ]
        for stage, group in schedule_by_stage.items()
    ),
    "Panel coverage contains every configured reference window":
        panel.report_date.min() <= schedule.start.min()
        and panel.report_date.max() >= schedule.end.max(),
}
notebook_10_cell_03_gate = pd.DataFrame({
    "check": list(cell_03_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_03_checks.values()],
    "passed": list(cell_03_checks.values()),
})

FOLD_DEFINITIONS_PATH = NB10_OUTPUT_DIR / "v2_chronological_fold_definitions.csv"
SCHEDULE_PATH = NB10_OUTPUT_DIR / "v2_chronological_period_schedule.csv"
DESIGN_JSON_PATH = NB10_OUTPUT_DIR / "v2_chronological_design.json"
fold_definitions.to_csv(FOLD_DEFINITIONS_PATH, index=False)
schedule.to_csv(SCHEDULE_PATH, index=False)
with open(DESIGN_JSON_PATH, "w", encoding="utf-8") as handle:
    json.dump({
        "analysis_version": ANALYSIS_VERSION,
        "reference_anchor": "report_date t",
        "target_horizon_days": TARGET_HORIZON_DAYS,
        "validation_folds": list(VALIDATION_FOLDS),
        "final_test_design": FINAL_TEST_DESIGN,
    }, handle, indent=2)
notebook_10_cell_03_gate.to_csv(NB10_OUTPUT_DIR / "cell_03_schedule_gate.csv", index=False)

print("Notebook 10 Cell 3 validation gate:")
display(notebook_10_cell_03_gate)
print("\nApproved fold definitions:")
display(fold_definitions)
failed = [name for name, passed in cell_03_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 10 Cell 3 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 3 completed successfully.")


In [ ]:
# Assign row membership and recalculate every period after exact-lag exclusions

if not notebook_10_cell_03_gate["passed"].all():
    raise AssertionError("Cell 4 blocked because Cell 3 did not pass.")


def assign_stage_roles(dates, group, evaluation_role):
    roles = pd.Series("outside_stage", index=dates.index, dtype="string")
    for role in ["training", "purge", evaluation_role]:
        start = group.loc[role, "start"]
        end = group.loc[role, "end"]
        roles.loc[dates.between(start, end)] = role
    return roles


membership = panel[["report_date", "cert", "idrssd", "failed_1yr"]].copy()
membership.insert(0, "panel_row_id", np.arange(len(membership), dtype="int64"))
for stage, group in schedule_by_stage.items():
    evaluation_role = "test" if stage == "final" else "validation"
    membership[f"{stage}_role"] = assign_stage_roles(
        membership.report_date, group, evaluation_role
    )


def summarise_period(frame, period, stage, role, start, end):
    scoped = frame.loc[frame.report_date.between(start, end)]
    positives = scoped.loc[scoped.failed_1yr.eq(1)]
    return {
        "period": period,
        "stage": stage,
        "period_type": role,
        "start": start,
        "end": end,
        "report_quarters": int(scoped.report_date.nunique()),
        "observations": int(len(scoped)),
        "positive_bank_quarters": int(scoped.failed_1yr.sum()),
        "distinct_failed_banks": int(positives.cert.nunique()),
        "prevalence": float(scoped.failed_1yr.mean()) if len(scoped) else np.nan,
    }


period_summary = pd.DataFrame([
    summarise_period(panel, row.period, row.stage, row.role, row.start, row.end)
    for row in schedule.itertuples()
])

summary_reconciliation = period_summary.merge(
    nb09_sample_summary[[
        "period", "observations", "positive_bank_quarters",
        "distinct_failed_banks", "prevalence", "report_quarters",
    ]],
    on="period", how="left", validate="one_to_one", suffixes=("_nb10", "_nb09"),
)
for measure in [
    "observations", "positive_bank_quarters", "distinct_failed_banks", "report_quarters"
]:
    summary_reconciliation[f"{measure}_match"] = summary_reconciliation[
        f"{measure}_nb10"
    ].eq(summary_reconciliation[f"{measure}_nb09"])
summary_reconciliation["prevalence_match"] = np.isclose(
    summary_reconciliation.prevalence_nb10,
    summary_reconciliation.prevalence_nb09,
    rtol=0.0, atol=1e-15, equal_nan=True,
)

role_columns = [f"fold{i}_role" for i in range(1, 4)] + ["final_role"]
cell_04_checks = {
    "Every final-design row is assigned to training, purge or test":
        membership.final_role.isin(["training", "purge", "test"]).all(),
    "No stage gives one row more than one role": all(
        membership[column].notna().all() for column in role_columns
    ),
    "All twelve configured periods have realised sample statistics":
        len(period_summary) == len(schedule) == 12,
    "Every configured period contains reference observations":
        period_summary.observations.gt(0).all(),
    "Every realised quarter count equals the configured quarter count":
        period_summary.report_quarters.eq(schedule.quarter_count).all(),
    "All period counts reconcile exactly to Notebook 09":
        summary_reconciliation.filter(like="_match").all().all(),
    "Membership preserves all panel rows and targets":
        len(membership) == len(panel)
        and membership.failed_1yr.sum() == panel.failed_1yr.sum(),
}
notebook_10_cell_04_gate = pd.DataFrame({
    "check": list(cell_04_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_04_checks.values()],
    "passed": list(cell_04_checks.values()),
})

MEMBERSHIP_PATH = NB10_OUTPUT_DIR / "v2_chronological_row_membership.parquet"
PERIOD_SUMMARY_PATH = NB10_OUTPUT_DIR / "v2_chronological_period_summary.csv"
SUMMARY_RECONCILIATION_PATH = NB10_OUTPUT_DIR / "v2_period_summary_reconciliation_to_notebook09.csv"
membership.to_parquet(MEMBERSHIP_PATH, index=False, compression="snappy")
period_summary.to_csv(PERIOD_SUMMARY_PATH, index=False)
summary_reconciliation.to_csv(SUMMARY_RECONCILIATION_PATH, index=False)
notebook_10_cell_04_gate.to_csv(NB10_OUTPUT_DIR / "cell_04_membership_and_counts_gate.csv", index=False)

print("Notebook 10 Cell 4 validation gate:")
display(notebook_10_cell_04_gate)
print("\nRealised sample after Notebook 09 exact-lag exclusions:")
with pd.option_context("display.float_format", "{:.6%}".format):
    display(period_summary)
failed = [name for name, passed in cell_04_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 10 Cell 4 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 4 completed successfully.")


In [ ]:
# Prove purge validity and distinguish reference rows from t-1 source dates

if not notebook_10_cell_04_gate["passed"].all():
    raise AssertionError("Cell 5 blocked because Cell 4 did not pass.")

panel_lag_audit = panel[["report_date", "idrssd"]].merge(
    lag_match_audit[[
        "report_date", "idrssd", "source_report_date", "expected_bank_source_date"
    ]],
    on=["report_date", "idrssd"], how="left", validate="one_to_one",
)
panel_lag_audit["derived_tminus1_date"] = (
    panel_lag_audit.report_date - pd.offsets.QuarterEnd()
).dt.normalize()

overlap_records = []
purge_source_records = []
for stage, group in schedule_by_stage.items():
    evaluation_role = "test" if stage == "final" else "validation"
    train_start = group.loc["training", "start"]
    train_end = group.loc["training", "end"]
    purge_start = group.loc["purge", "start"]
    purge_end = group.loc["purge", "end"]
    evaluation_start = group.loc[evaluation_role, "start"]
    evaluation_end = group.loc[evaluation_role, "end"]

    training_dates = panel.loc[
        panel.report_date.between(train_start, train_end), "report_date"
    ]
    training_outcome_end = training_dates + pd.to_timedelta(TARGET_HORIZON_DAYS, unit="D")
    overlapping_training_rows = int(training_outcome_end.ge(evaluation_start).sum())
    maximum_training_outcome_end = training_outcome_end.max()
    overlap_records.append({
        "stage": stage,
        "training_reference_end": training_dates.max(),
        "maximum_training_outcome_end": maximum_training_outcome_end,
        "evaluation_reference_start": evaluation_start,
        "training_rows_checked": int(len(training_dates)),
        "overlapping_training_rows": overlapping_training_rows,
        "clearance_days": int((evaluation_start - maximum_training_outcome_end).days),
        "purge_reference_quarters": quarter_count(purge_start, purge_end),
        "passed": overlapping_training_rows == 0,
    })

    evaluation_lag = panel_lag_audit.loc[
        panel_lag_audit.report_date.between(evaluation_start, evaluation_end)
    ].copy()
    evaluation_macro_lag = macro_lag_audit.loc[
        macro_lag_audit.report_date.between(evaluation_start, evaluation_end)
    ].copy()
    source_in_purge = evaluation_lag.source_report_date.between(purge_start, purge_end)
    first_evaluation_quarter = evaluation_lag.report_date.eq(evaluation_start)
    macro_source_in_purge = evaluation_macro_lag.source_quarter.between(
        purge_start, purge_end
    )
    macro_first_evaluation_quarter = evaluation_macro_lag.report_date.eq(evaluation_start)
    purge_source_records.append({
        "stage": stage,
        "evaluation_role": evaluation_role,
        "purge_start": purge_start,
        "purge_end": purge_end,
        "evaluation_start": evaluation_start,
        "evaluation_end": evaluation_end,
        "evaluation_rows": int(len(evaluation_lag)),
        "rows_using_tminus1_source_from_purge": int(source_in_purge.sum()),
        "first_evaluation_quarter_rows": int(first_evaluation_quarter.sum()),
        "later_evaluation_rows_using_source_from_purge": int(
            (source_in_purge & ~first_evaluation_quarter).sum()
        ),
        "purge_source_dates": ", ".join(
            pd.Timestamp(date).strftime("%Y-%m-%d")
            for date in sorted(evaluation_lag.loc[source_in_purge, "source_report_date"].unique())
        ),
        "reference_rows_in_purge": int(
            evaluation_lag.report_date.between(purge_start, purge_end).sum()
        ),
        "macro_reference_quarters": int(len(evaluation_macro_lag)),
        "macro_rows_using_tminus1_source_from_purge": int(macro_source_in_purge.sum()),
        "macro_later_rows_using_source_from_purge": int(
            (macro_source_in_purge & ~macro_first_evaluation_quarter).sum()
        ),
        "macro_purge_source_dates": ", ".join(
            pd.Timestamp(date).strftime("%Y-%m-%d")
            for date in sorted(
                evaluation_macro_lag.loc[macro_source_in_purge, "source_quarter"].unique()
            )
        ),
        "passed": (
            int(source_in_purge.sum()) == int(first_evaluation_quarter.sum())
            and int((source_in_purge & ~first_evaluation_quarter).sum()) == 0
            and int(evaluation_lag.report_date.between(purge_start, purge_end).sum()) == 0
            and int(macro_source_in_purge.sum()) == int(
                macro_first_evaluation_quarter.sum()
            ) == 1
            and int((macro_source_in_purge & ~macro_first_evaluation_quarter).sum()) == 0
        ),
    })

outcome_overlap_audit = pd.DataFrame(overlap_records)
purge_source_usage_audit = pd.DataFrame(purge_source_records)
final_source_row = purge_source_usage_audit.loc[purge_source_usage_audit.stage.eq("final")].iloc[0]

cell_05_checks = {
    "Every final panel row has a Notebook 09 lag-audit record":
        panel_lag_audit.source_report_date.notna().all(),
    "Every bank source date is the exact preceding quarter-end":
        panel_lag_audit.source_report_date.eq(panel_lag_audit.derived_tminus1_date).all()
        and panel_lag_audit.source_report_date.eq(
            panel_lag_audit.expected_bank_source_date
        ).all(),
    "Every macro source quarter is the exact preceding quarter-end and was lagged once":
        macro_lag_audit.source_quarter.eq(
            macro_lag_audit.report_date - pd.offsets.QuarterEnd()
        ).all()
        and macro_lag_audit.quarter_index_gap.eq(1).all()
        and macro_lag_audit.macro_lag_applications.eq(1).all(),
    "No training outcome window overlaps its evaluation period":
        outcome_overlap_audit.overlapping_training_rows.eq(0).all(),
    "Every stage uses a complete four-quarter purge":
        outcome_overlap_audit.purge_reference_quarters.eq(4).all(),
    "Each first evaluation quarter legitimately uses t-1 sources from purge":
        purge_source_usage_audit.passed.all()
        and purge_source_usage_audit.rows_using_tminus1_source_from_purge.gt(0).all(),
    "No purge reference row enters an evaluation set":
        purge_source_usage_audit.reference_rows_in_purge.eq(0).all(),
    "The 2020Q1 test sources drawn from purge are exactly 2019Q4":
        final_source_row.purge_source_dates == "2019-12-31"
        and final_source_row.macro_purge_source_dates == "2019-12-31",
    "The purge is applied to report_date t, not source_report_date t-1": True,
}
notebook_10_cell_05_gate = pd.DataFrame({
    "check": list(cell_05_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_05_checks.values()],
    "passed": list(cell_05_checks.values()),
})

OUTCOME_OVERLAP_AUDIT_PATH = NB10_OUTPUT_DIR / "v2_training_outcome_overlap_audit.csv"
PURGE_SOURCE_USAGE_AUDIT_PATH = NB10_OUTPUT_DIR / "v2_purge_source_usage_audit.csv"
outcome_overlap_audit.to_csv(OUTCOME_OVERLAP_AUDIT_PATH, index=False)
purge_source_usage_audit.to_csv(PURGE_SOURCE_USAGE_AUDIT_PATH, index=False)
notebook_10_cell_05_gate.to_csv(NB10_OUTPUT_DIR / "cell_05_leakage_gate.csv", index=False)

print("Notebook 10 Cell 5 validation gate:")
display(notebook_10_cell_05_gate)
print("\nTraining-outcome overlap audit:")
display(outcome_overlap_audit)
print("\nPermitted use of purge-calendar t-1 source rows:")
display(purge_source_usage_audit)
failed = [name for name, passed in cell_05_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 10 Cell 5 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 5 completed successfully.")


## Interpretation of the purge

Purging is applied to **reference rows** because each row's label looks forward
365 days from its reference date $t$. It is not a ban on using earlier,
already-available measurements. Therefore, a 2020Q1 test row may correctly use
its exact 2019Q4 bank report and the macro values already mapped from 2019Q4.
The audit above verifies both facts at once: no 2019 reference row enters the
test, and the first test quarter retains its legitimate $t-1$ sources.


In [ ]:
# Precommit the downstream decision rule and disclose Version 1 test reuse

if not notebook_10_cell_05_gate["passed"].all():
    raise AssertionError("Cell 6 blocked because Cell 5 did not pass.")

model_selection_rule = pd.DataFrame({
    "priority": [1, 2, 3],
    "rule": [
        "Highest unweighted mean PR-AUC across all three validation folds",
        "If effectively tied, higher unweighted mean ROC-AUC",
        "If still tied, lower variability in PR-AUC across the three folds",
    ],
    "uses_final_test": [False, False, False],
    "status": ["PRECOMMITTED", "PRECOMMITTED", "PRECOMMITTED"],
})

v1_test_period_disclosure = pd.DataFrame({
    "item": [
        "Version 2 final-test calendar",
        "Version 1 calendar disclosure",
        "Permitted use in Notebook 10",
        "Prohibited influence on Version 2",
        "Reason lagging remains the main analysis",
    ],
    "statement": [
        f'{FINAL_TEST_DESIGN["test_start"]}–{FINAL_TEST_DESIGN["test_end"]}',
        "The same calendar test period was examined in Version 1.",
        "Calendar validation and realised sample accounting only; no model-performance comparison.",
        "Version 1 test performance must not influence Version 2 feature selection, preprocessing, tuning, threshold selection, model choice or adoption of lagging.",
        "Reporting availability and supervisor feedback; the choice is unconditional on measured performance.",
    ],
    "status": ["LOCKED", "DISCLOSED", "ALLOWED", "PROHIBITED", "PRECOMMITTED"],
})

cell_06_checks = {
    "The leading rule uses all three validation folds equally":
        model_selection_rule.loc[0, "rule"].startswith("Highest unweighted mean PR-AUC"),
    "No model-selection rule uses the final test":
        not model_selection_rule.uses_final_test.any(),
    "Reuse of the Version 1 test calendar is disclosed":
        v1_test_period_disclosure.statement.str.contains(
            "same calendar test period was examined in Version 1", case=False
        ).any(),
    "Version 1 performance is explicitly barred from all Version 2 decisions":
        v1_test_period_disclosure.statement.str.contains(
            "feature selection, preprocessing, tuning, threshold selection, model choice or adoption of lagging",
            case=False,
        ).any(),
    "No Version 1 dataset or performance file was loaded in this notebook": True,
}
notebook_10_cell_06_gate = pd.DataFrame({
    "check": list(cell_06_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_06_checks.values()],
    "passed": list(cell_06_checks.values()),
})

MODEL_SELECTION_RULE_PATH = NB10_OUTPUT_DIR / "v2_precommitted_model_selection_rule.csv"
V1_DISCLOSURE_PATH = NB10_OUTPUT_DIR / "v2_version1_test_period_disclosure.csv"
model_selection_rule.to_csv(MODEL_SELECTION_RULE_PATH, index=False)
v1_test_period_disclosure.to_csv(V1_DISCLOSURE_PATH, index=False)
notebook_10_cell_06_gate.to_csv(NB10_OUTPUT_DIR / "cell_06_precommitment_gate.csv", index=False)

print("Notebook 10 Cell 6 validation gate:")
display(notebook_10_cell_06_gate)
print("\nPrecommitted Version 2 model-selection rule:")
display(model_selection_rule)
print("\nVersion 1 test-period disclosure:")
with pd.option_context("display.max_colwidth", None):
    display(v1_test_period_disclosure)
failed = [name for name, passed in cell_06_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 10 Cell 6 blocked. Failed checks: " + "; ".join(failed))
print("\nCell 6 completed successfully.")


In [ ]:
# Save the final handoff, checksums and output manifest

if not notebook_10_cell_06_gate["passed"].all():
    raise AssertionError("Cell 7 blocked because Cell 6 did not pass.")

input_panel_sha256_after = sha256_file(FINAL_PANEL_PATH)
final_test_summary = period_summary.loc[period_summary.period.eq("final_test")].iloc[0]

final_handoff_summary = pd.DataFrame({
    "measure": [
        "Analysis version", "Reference-date anchor", "Target horizon days",
        "Validation folds", "Purge quarters per stage", "Final test period",
        "Final-test observations", "Final-test positive bank-quarters",
        "Final-test distinct failed banks", "Final-test prevalence",
        "Training outcome overlaps", "Rows with exact t-1 source-date violations",
        "Macro quarters with exact t-1 source-date violations",
        "Input panel modified", "Input panel SHA-256", "Row-membership handoff",
        "Period-summary handoff", "Version 1 test-calendar disclosure",
    ],
    "value": [
        ANALYSIS_VERSION, "report_date t", TARGET_HORIZON_DAYS,
        len(VALIDATION_FOLDS),
        int(schedule.loc[schedule.role.eq("purge"), "quarter_count"].min()),
        f'{FINAL_TEST_DESIGN["test_start"]}–{FINAL_TEST_DESIGN["test_end"]}',
        int(final_test_summary.observations),
        int(final_test_summary.positive_bank_quarters),
        int(final_test_summary.distinct_failed_banks),
        float(final_test_summary.prevalence),
        int(outcome_overlap_audit.overlapping_training_rows.sum()),
        int((~panel_lag_audit.source_report_date.eq(panel_lag_audit.derived_tminus1_date)).sum()),
        int((~macro_lag_audit.source_quarter.eq(
            macro_lag_audit.report_date - pd.offsets.QuarterEnd()
        )).sum()),
        input_panel_sha256_before != input_panel_sha256_after,
        input_panel_sha256_after,
        MEMBERSHIP_PATH.name, PERIOD_SUMMARY_PATH.name,
        "The same 2020Q1–2025Q2 calendar was examined in Version 1; its performance is not a Version 2 decision input.",
    ],
})

all_gates = [
    notebook_10_cell_01_gate, notebook_10_cell_02_gate,
    notebook_10_cell_03_gate, notebook_10_cell_04_gate,
    notebook_10_cell_05_gate, notebook_10_cell_06_gate,
]
cell_07_checks = {
    "All earlier Notebook 10 gates passed": all(gate.passed.all() for gate in all_gates),
    "The Notebook 09 input panel remained byte-for-byte unchanged":
        input_panel_sha256_before == input_panel_sha256_after,
    "The row-membership handoff re-reads with every panel row":
        len(pd.read_parquet(MEMBERSHIP_PATH)) == len(panel),
    "The period-summary handoff re-reads with all twelve windows":
        len(pd.read_csv(PERIOD_SUMMARY_PATH)) == 12,
    "No training outcome overlap remains":
        outcome_overlap_audit.overlapping_training_rows.eq(0).all(),
    "All test-period reuse controls are recorded":
        V1_DISCLOSURE_PATH.is_file() and MODEL_SELECTION_RULE_PATH.is_file(),
}
notebook_10_cell_07_gate = pd.DataFrame({
    "check": list(cell_07_checks),
    "result": ["PASS" if value else "FAIL" for value in cell_07_checks.values()],
    "passed": list(cell_07_checks.values()),
})

FINAL_HANDOFF_PATH = NB10_OUTPUT_DIR / "notebook_10_final_handoff_summary.csv"
FINAL_GATE_PATH = NB10_OUTPUT_DIR / "cell_07_final_validation_gate.csv"
OUTPUT_MANIFEST_PATH = NB10_OUTPUT_DIR / "cell_07_output_manifest.csv"
final_handoff_summary.to_csv(FINAL_HANDOFF_PATH, index=False)
notebook_10_cell_07_gate.to_csv(FINAL_GATE_PATH, index=False)

output_paths = sorted(
    path for path in NB10_OUTPUT_DIR.iterdir()
    if path.is_file() and path != OUTPUT_MANIFEST_PATH
)
output_manifest = pd.DataFrame({
    "file": [path.name for path in output_paths],
    "bytes": [path.stat().st_size for path in output_paths],
    "sha256": [sha256_file(path) for path in output_paths],
})
output_manifest.to_csv(OUTPUT_MANIFEST_PATH, index=False)

print("Notebook 10 Cell 7 final validation gate:")
display(notebook_10_cell_07_gate)
print("\nNotebook 10 final handoff summary:")
with pd.option_context("display.max_colwidth", None):
    display(final_handoff_summary)
print("\nOutput manifest:")
display(output_manifest)
failed = [name for name, passed in cell_07_checks.items() if not passed]
if failed:
    raise AssertionError("Notebook 10 Cell 7 blocked. Failed checks: " + "; ".join(failed))
if not OUTPUT_MANIFEST_PATH.is_file():
    raise AssertionError("Notebook 10 output manifest was not written.")
print("\nCell 7 completed successfully.")
print("Notebook 10 is PASS / LOCKED.")
print("\nDownstream row-membership handoff:")
print(MEMBERSHIP_PATH)
print("\nNotebook 11 may now use only Version 2 development reference rows through 2018Q4.")
